<a href="https://colab.research.google.com/github/Savindi2002/CV-Analyzer-Job-Matching/blob/main/CV_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn nltk beautifulsoup4 sentence-transformers transformers torch

In [4]:
import pandas as pd
import numpy as np
import re
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
resume_path = "/content/drive/MyDrive/NLP Project/Resume.csv"
job_path = "/content/drive/MyDrive/NLP Project/data.csv"

In [8]:
resume_df = pd.read_csv(resume_path)
job_df = pd.read_csv(job_path)

In [9]:
print("Resume Dataset:")
print(resume_df.shape)

print("\nJob Dataset:")
print(job_df.shape)

Resume Dataset:
(2484, 4)

Job Dataset:
(521, 2)


In [10]:
print("Resume columns:")
print(resume_df.columns.tolist())

print("\nJob columns:")
print(job_df.columns.tolist())

Resume columns:
['ID', 'Resume_str', 'Resume_html', 'Category']

Job columns:
['Job Title', 'Description']


In [11]:
resume_df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [12]:
job_df.head()

,Job Title,Description
0,Data Analyst,job overview were seeking a data analyst to tu...
1,Data Reporting Analyst,about wspc wspc is a cooperative of outstandin...
2,Data Analyst (Power BI/Python),data analyst power bipython employment type fu...
3,Data & Reporting Analyst,our company pharmerica overview the data repor...
4,Data Quality Analyst (Remote Opportunity),vetsez is seeking a data quality analyst teste...


In [13]:
print("Resume missing values:")
print(resume_df.isnull().sum())

print("\nJob missing values:")
print(job_df.isnull().sum())

Resume missing values:
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

Job missing values:
Job Title      0
Description    0
dtype: int64


In [14]:
print("Duplicate resumes:", resume_df.duplicated().sum())
print("Duplicate jobs:", job_df.duplicated().sum())

Duplicate resumes: 0
Duplicate jobs: 0


In [15]:
resume_df = resume_df.drop_duplicates()
job_df = job_df.drop_duplicates()

In [16]:
print(resume_df.columns.tolist())

['ID', 'Resume_str', 'Resume_html', 'Category']


In [18]:
resume_df = resume_df[['ID', 'Resume_str', 'Category']]

resume_df.head()
resume_df.tail()

,ID,Resume_str,Category
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,AVIATION
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...",AVIATION
2481,31605080,GEEK SQUAD AGENT Professional...,AVIATION
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,AVIATION
2483,37473139,STOREKEEPER II Professional Sum...,AVIATION


In [19]:
resume_df = resume_df.rename(columns={
    'ID': 'resume_id',
    'Resume_str': 'resume_text',
    'Category': 'category'
})

job_df = job_df.rename(columns={
    'Job Title': 'job_title',
    'Description': 'job_description'
})

In [20]:
print(resume_df.columns)
print(job_df.columns)

Index(['resume_id', 'resume_text', 'category'], dtype='object')
Index(['job_title', 'job_description'], dtype='object')


In [21]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text)

    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [22]:
resume_df['clean_resume'] = resume_df['resume_text'].apply(clean_text)

job_df['clean_job'] = job_df['job_description'].apply(clean_text)

In [23]:
print("BEFORE:")
print(resume_df['resume_text'].iloc[0][:1000])

print("\n\nAFTER:")
print(resume_df['clean_resume'].iloc[0][:1000])

BEFORE:
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, lo

In [24]:
sample_text = resume_df['clean_resume'].iloc[0]

tokens = sample_text.split()

print(tokens[:30])

['hr', 'administrator', 'marketing', 'associate', 'hr', 'administrator', 'summary', 'dedicated', 'customer', 'service', 'manager', 'with', '15', 'years', 'of', 'experience', 'in', 'hospitality', 'and', 'customer', 'service', 'management', 'respected', 'builder', 'and', 'leader', 'of', 'customer', 'focused', 'teams']
